In [21]:
#Source : https://github.com/PacktPublishing/Python-Natural-Language-Processing-Cookbook-Second-Edition/tree/main/data

In [22]:
# !python -m spacy download en_core_web_lg


In [23]:
import nltk
import spacy
from nltk import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
small_model = spacy.load("en_core_web_sm")
large_model = spacy.load("en_core_web_lg")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lwhitenack/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Basic method: spaCy lemma versus spaCy token

In [24]:
text = "I have five birds"
doc = small_model(text)
for token in doc:
    if (token.pos_ == "NOUN" and token.lemma_ != token.text):
        print(token.text, "plural")

birds plural


# Number using morph features

In [25]:
doc = small_model("I have five birds.")
print(doc[3].morph.get("Number"))

['Plur']


# Function to determine number using spaCy

In [ ]:
from enum import Enum


In [27]:
class Noun_number(Enum):
    SINGULAR = 1
    PLURAL = 2

def get_nouns_number(text, model, method="lemma"):
    nouns = []
    doc = model(text)
    for token in doc:
        if (token.pos_ == "NOUN"):
            if method == "lemma":
                if token.lemma_ != token.text:
                    nouns.append((token.text, Noun_number.PLURAL))
                else:
                    nouns.append((token.text, Noun_number.SINGULAR))
            elif method == "morph":
                if token.morph.get("Number") == "Sing":
                    nouns.append((token.text, Noun_number.PLURAL))
                else:
                    nouns.append((token.text, Noun_number.SINGULAR))
    return nouns

# Irregular nouns using small model

In [28]:
text = "Three geese crossed the road"
nouns = get_nouns_number(text, small_model, "morph")
print(nouns)
nouns = get_nouns_number(text, small_model)
print(nouns)

[('geese', <Noun_number.SINGULAR: 1>), ('road', <Noun_number.SINGULAR: 1>)]
[('geese', <Noun_number.PLURAL: 2>), ('road', <Noun_number.SINGULAR: 1>)]


# Irregular nouns using large model

In [29]:
#!python -m spacy download en_core_web_lg
large_model = spacy.load("en_core_web_lg")
nouns = get_nouns_number(text, large_model, "morph")
print(nouns)
nouns = get_nouns_number(text, large_model)
print(nouns)


[('geese', <Noun_number.SINGULAR: 1>), ('road', <Noun_number.SINGULAR: 1>)]
[('geese', <Noun_number.PLURAL: 2>), ('road', <Noun_number.SINGULAR: 1>)]


# Noun number using GPT-3

In [30]:
from openai import OpenAI
client = OpenAI(api_key=OPEN_AI_KEY)
prompt="""Decide whether each noun in the following text is singular or plural.
Return the list in the format of a python tuple: (word, number). Do not provide any additional explanations.
Sentence: Three geese crossed the road."""
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=0,
    max_tokens=256,
    top_p=1.0,
    frequency_penalty=0,
    presence_penalty=0,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ],
)
print(response.choices[0].message.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# Converting from singular to plural and plural to singular

In [ ]:
from textblob import TextBlob
texts = ["book", "goose", "pen", "point", "deer"]
blob_objs = [TextBlob(text) for text in texts]
plurals = [blob_obj.words.pluralize()[0] for blob_obj in blob_objs]
print(plurals)
blob_objs = [TextBlob(text) for text in plurals]
singulars = [blob_obj.words.singularize()[0] for blob_obj in blob_objs]
print(singulars)

['books', 'geese', 'pens', 'points', 'deer']
['book', 'goose', 'pen', 'point', 'deer']


This code snippet uses spaCy to identify plural nouns in the text "I have five birds". Here's a breakdown:

1.  **`text = "I have five birds"`**: This line defines the input text string.
2.  **`doc = small_model(text)`**: This line processes the input text using the small spaCy model (`en_core_web_sm`). spaCy tokenizes the text and performs various linguistic analyses, storing the results in a `doc` object.
3.  **`for token in doc:`**: This loop iterates through each token (word or punctuation) in the processed `doc`.
4.  **`if (token.pos_ == "NOUN" and token.lemma_ != token.text):`**: This is a conditional statement that checks two conditions for each token:
    *   **`token.pos_ == "NOUN"`**: This checks if the token's part-of-speech tag is "NOUN".
    *   **`token.lemma_ != token.text`**: This checks if the token's lemma (base form) is different from its original text. For regular plural nouns, the lemma will be the singular form (e.g., "bird" for "birds"), so this condition will be true. This is a simple way to identify potential plural nouns based on whether their form differs from their base form.
5.  **`print(token.text, "plural")`**: If both conditions in the `if` statement are true, this line prints the original text of the token followed by the word "plural". In this case, for the token "birds", the conditions are met, and the output is "birds plural".